In [1]:
pip install -q langgraph langchain-google-genai langchain-core pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, json, re, pickle
from typing import TypedDict, List, Dict, Any, Optional
from collections import defaultdict
import pandas as pd

# ============ GEMINI KEY ============
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    os.environ.setdefault("GEMINI_API_KEY", "")

MODEL_NAME = "gemini-3.8-flash"   

# ============ DATASET PATHS ============
PATH = " "   
F_PLAYER_INFO  = PATH + "afl_players_info_raw.csv"
F_ROUND_STATS  = PATH + "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"
F_SEASON_STATS = PATH + "afl_players_seasonal_stats_raw.csvafl_players_seasonal_stats_raw.csv"
F_TEAM_MATCHES = PATH + "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv"

print("Key set:", bool(os.environ.get("GEMINI_API_KEY")))

Key set: True


In [2]:
player_info  = pd.read_csv("afl_players_info_raw.csv")
round_stats  = pd.read_csv("afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv")
season_stats = pd.read_csv("afl_players_seasonal_stats_raw.csv", low_memory=False)
team_matches = pd.read_csv("team_matches_home_away_raw - team_matches_home_away_raw.csv.csv")

team_matches["match_date"] = pd.to_datetime(team_matches["match_date"], errors="coerce")

print("player_info :", player_info.shape)
print("round_stats :", round_stats.shape)
print("season_stats:", season_stats.shape)
print("team_matches:", team_matches.shape)
print()
print("player_info cols:", list(player_info.columns))
print()
print("round_stats cols:", list(round_stats.columns))
print()
print("team_matches cols:", list(team_matches.columns))

player_info : (2848, 16)
round_stats : (274089, 36)
season_stats: (25491, 54)
team_matches: (15808, 19)

player_info cols: ['id', 'player_name', 'player_full_name', 'first_name', 'last_name', 'born_date', 'debut_date', 'debut_age', 'last_date', 'last_age', 'height', 'weight', 'profile_pic', 'player_link', 'player_common_names', 'player_teams']

round_stats cols: ['id', 'team', 'year', 'career_game_count', 'opponent', 'round', 'result', 'jersey_num', 'kicks', 'marks', 'handballs', 'disposals', 'goals', 'behinds', 'hit_outs', 'tackles', 'rebound_50s', 'inside_50s', 'clearances', 'clangers', 'free_kicks_for', 'free_kicks_against', 'brownlow_votes', 'contested_possessions', 'uncontested_possessions', 'contested_marks', 'marks_inside_50', 'one_percenters', 'bounces', 'goal_assist', 'percentage_of_game_played', 'player_id', 'match_date', 'fantasy_points', 'score', 'margin']

team_matches cols: ['id', 'team_name', 'round', 'match_date', 'year', 'home_away', 'opponent', 'team_quarter_scores', 

In [3]:
# Cell 4 (FINAL): player_id se naam map karo + underscore hatao

# 1. Purana galat player_name column drop karo
if "player_name" in round_stats.columns:
    round_stats = round_stats.drop(columns=["player_name"])

# 2. player_info se id → naam ka mapping
id_to_name = player_info.set_index("id")["player_full_name"].to_dict()

# 3. Map karo
round_stats["player_name"] = round_stats["player_id"].map(id_to_name)

# 4. Cosmetic fix: "Mark_Graham" → "Mark Graham"
round_stats["player_name"] = round_stats["player_name"].str.replace("_", " ", regex=False)

# 5. Khaali naam wali rows drop karo (chhoti si safai)
round_stats = round_stats.dropna(subset=["player_name"]).copy()

print("Final round_stats shape:", round_stats.shape)
print("player_name non-null :", round_stats["player_name"].notna().sum())
print()
print(round_stats[["player_id","player_name","team","year","goals"]].head(10))

Final round_stats shape: (271810, 37)
player_name non-null : 271810

   player_id     player_name                 team  year  goals
0      45552     Mark Graham       Hawthorn Hawks  1994    NaN
1      44356   Shaun Mannagh         Geelong Cats  2024    NaN
2      45955    Sean Wellman     Essendon Bombers  1999    NaN
3      45656  Steven Kretiuk     Western Bulldogs  1994    NaN
4      45929    Scott Turner      Richmond Tigers  1997    NaN
5      45344    James Begley      St Kilda Saints  2002    NaN
6      45630   Anthony Jones    Fremantle Dockers  2002    NaN
7      45747     Mal Michael       Brisbane Lions  2002    NaN
8      45630   Anthony Jones    Fremantle Dockers  1997    NaN
9      45591    Adam Heuskes  Port Adelaide Power  1997    NaN


In [4]:
# Cell 5 (FIXED): Team names clean karo + aliases banao

# 1. team_matches mein team_name clean karo (whitespace + tab hatao)
team_matches["team_name"] = team_matches["team_name"].astype(str).str.strip()
team_matches["opponent"]  = team_matches["opponent"].astype(str).str.strip()

# 2. round_stats mein bhi team clean karo
round_stats["team"]     = round_stats["team"].astype(str).str.strip()
round_stats["opponent"] = round_stats["opponent"].astype(str).str.strip()

# 3. Ab KNOWN_TEAMS nikaalo (clean)
KNOWN_TEAMS = sorted(set(team_matches["team_name"].dropna().unique()))
print(f"Total teams: {len(KNOWN_TEAMS)}")
for t in KNOWN_TEAMS:
    print(f"  - {t!r}")

print()

# 4. Alias map — dataset ke EXACT names ke saath
TEAM_ALIASES = {
    # Adelaide
    "crows":"Adelaide Crows","adelaide":"Adelaide Crows","adelaide crows":"Adelaide Crows",
    # Brisbane
    "lions":"Brisbane Lions","brisbane":"Brisbane Lions","brisbane lions":"Brisbane Lions",
    "bears":"Brisbane Bears","brisbane bears":"Brisbane Bears",
    # Carlton
    "blues":"Carlton Blues","carlton":"Carlton Blues","carlton blues":"Carlton Blues",
    # Collingwood
    "pies":"Collingwood Magpies","magpies":"Collingwood Magpies",
    "collingwood":"Collingwood Magpies","collingwood magpies":"Collingwood Magpies",
    # Essendon
    "bombers":"Essendon Bombers","essendon":"Essendon Bombers","essendon bombers":"Essendon Bombers",
    # Fitzroy (historical)
    "fitzroy":"Fitzroy Lions","fitzroy lions":"Fitzroy Lions",
    # Fremantle
    "dockers":"Fremantle Dockers","freo":"Fremantle Dockers",
    "fremantle":"Fremantle Dockers","fremantle dockers":"Fremantle Dockers",
    # Geelong
    "cats":"Geelong Cats","geelong":"Geelong Cats","geelong cats":"Geelong Cats",
    # Gold Coast
    "suns":"Gold Coast Suns","gold coast":"Gold Coast Suns","gold coast suns":"Gold Coast Suns",
    # GWS
    "giants":"Greater Western Sydney Giants","gws":"Greater Western Sydney Giants",
    "gws giants":"Greater Western Sydney Giants",
    "greater western sydney":"Greater Western Sydney Giants",
    "greater western sydney giants":"Greater Western Sydney Giants",
    # Hawthorn
    "hawks":"Hawthorn Hawks","hawthorn":"Hawthorn Hawks","hawthorn hawks":"Hawthorn Hawks",
    # Melbourne
    "demons":"Melbourne Demons","dees":"Melbourne Demons",
    "melbourne":"Melbourne Demons","melbourne demons":"Melbourne Demons",
    # North Melbourne
    "kangaroos":"North Melbourne Kangaroos","roos":"North Melbourne Kangaroos",
    "north":"North Melbourne Kangaroos","north melbourne":"North Melbourne Kangaroos",
    "north melbourne kangaroos":"North Melbourne Kangaroos",
    # Port Adelaide
    "power":"Port Adelaide Power","port":"Port Adelaide Power",
    "port adelaide":"Port Adelaide Power","port adelaide power":"Port Adelaide Power",
    # Richmond
    "tigers":"Richmond Tigers","richmond":"Richmond Tigers","richmond tigers":"Richmond Tigers",
    # St Kilda
    "saints":"St Kilda Saints","st kilda":"St Kilda Saints","st kilda saints":"St Kilda Saints",
    # Sydney
    "swans":"Sydney Swans","sydney":"Sydney Swans","sydney swans":"Sydney Swans",
    # Western Bulldogs (dataset mein "W. Bulldogs" hai!)
    "bulldogs":"W. Bulldogs","dogs":"W. Bulldogs","footscray":"W. Bulldogs",
    "western bulldogs":"W. Bulldogs","w bulldogs":"W. Bulldogs","w. bulldogs":"W. Bulldogs",
    # West Coast
    "eagles":"West Coast Eagles","west coast":"West Coast Eagles",
    "west coast eagles":"West Coast Eagles",
}

def resolve_team(name):
    if not name: return None
    key = str(name).strip().lower()
    # Exact alias match
    if key in TEAM_ALIASES:
        cand = TEAM_ALIASES[key]
        if cand in KNOWN_TEAMS: return cand
    # Direct dataset name match
    for t in KNOWN_TEAMS:
        if t.lower() == key: return t
    # Partial match
    for t in KNOWN_TEAMS:
        if key in t.lower() or t.lower() in key: return t
    return None

# 5. Test
print("Alias test:")
for x in ["Pies","Cats","Hawks","Tigers","Swans","Bombers","Demons",
          "Bulldogs","Dogs","Western Bulldogs","Giants","GWS","Power","Suns"]:
    print(f"  {x:20s} -> {resolve_team(x)}")

Total teams: 20
  - 'Adelaide Crows'
  - 'Brisbane Bears'
  - 'Brisbane Lions'
  - 'Carlton Blues'
  - 'Collingwood Magpies'
  - 'Essendon Bombers'
  - 'Fitzroy Lions'
  - 'Fremantle Dockers'
  - 'Geelong Cats'
  - 'Gold Coast Suns'
  - 'Greater Western Sydney Giants'
  - 'Hawthorn Hawks'
  - 'Melbourne Demons'
  - 'North Melbourne Kangaroos'
  - 'Port Adelaide Power'
  - 'Richmond Tigers'
  - 'St Kilda Saints'
  - 'Sydney Swans'
  - 'W. Bulldogs'
  - 'West Coast Eagles'

Alias test:
  Pies                 -> Collingwood Magpies
  Cats                 -> Geelong Cats
  Hawks                -> Hawthorn Hawks
  Tigers               -> Richmond Tigers
  Swans                -> Sydney Swans
  Bombers              -> Essendon Bombers
  Demons               -> Melbourne Demons
  Bulldogs             -> W. Bulldogs
  Dogs                 -> W. Bulldogs
  Western Bulldogs     -> W. Bulldogs
  Giants               -> Greater Western Sydney Giants
  GWS                  -> Greater Western Sydney

In [5]:
# Cell 5.5 (quick sanity check after clean)

print("team_matches unique teams:", team_matches["team_name"].nunique())
print("round_stats  unique teams:", round_stats["team"].nunique())
print()
print("team_matches sample:")
print(team_matches[["team_name","opponent","match_date","team_score","opponent_score"]].head(3))
print()
print("round_stats sample:")
print(round_stats[["player_name","team","year","goals","disposals"]].head(3))

team_matches unique teams: 20
round_stats  unique teams: 20

team_matches sample:
                   team_name                   opponent match_date  \
0             Hawthorn Hawks  North Melbourne Kangaroos 1994-09-10   
1  North Melbourne Kangaroos             Hawthorn Hawks 1994-09-10   
2  North Melbourne Kangaroos             Brisbane Lions 2008-05-31   

   team_score  opponent_score  
0          91             114  
1         114              91  
2          98             129  

round_stats sample:
     player_name              team  year  goals  disposals
0    Mark Graham    Hawthorn Hawks  1994    NaN        2.0
1  Shaun Mannagh      Geelong Cats  2024    NaN        NaN
2   Sean Wellman  Essendon Bombers  1999    NaN       14.0


In [6]:
# Cell 6: Baseline prediction models

import pickle
from collections import defaultdict

MODEL_MATCH_PATH = "match_winner_model.pkl"
MODEL_TOP_PATH   = "top_player_model.pkl"

# ---------- Match winner ----------
if os.path.exists(MODEL_MATCH_PATH):
    predict_match_winner = pickle.load(open(MODEL_MATCH_PATH,"rb"))
    print("Loaded match model ✔")
else:
    print("[fallback] frequency-based match model")
    _win = defaultdict(int)
    for _, r in team_matches.iterrows():
        if pd.notna(r["team_score"]) and pd.notna(r["opponent_score"]):
            if r["team_score"] > r["opponent_score"]:
                _win[r["team_name"]] += 1
            elif r["opponent_score"] > r["team_score"]:
                _win[r["opponent"]] += 1
    _tot = sum(_win.values()) or 1

    def predict_match_winner(home_team, away_team, **kw):
        hp = _win.get(home_team,1)/_tot + 0.05
        ap = _win.get(away_team,1)/_tot
        s  = hp + ap
        return {"home_team":home_team,"away_team":away_team,
                "home_prob":round(hp/s,3),"away_prob":round(ap/s,3),
                "top_features":["historical win rate","home advantage"]}

# ---------- Top player ----------
if os.path.exists(MODEL_TOP_PATH):
    predict_top_player = pickle.load(open(MODEL_TOP_PATH,"rb"))
    print("Loaded top-player model ✔")
else:
    print("[fallback] avg-stat top-player model")
    def predict_top_player(team, stat="goals", **kw):
        sub = round_stats[round_stats["team"]==team].copy()
        if sub.empty: return None
        if stat not in sub.columns: stat = "goals"
        c = sub.groupby("player_name")[stat].mean().dropna().sort_values(ascending=False)
        if c.empty: return None
        return {"team":team,"stat":stat,
                "predictions":[(p,round(float(v),2)) for p,v in c.head(3).items()],
                "top_features":[f"avg {stat} last 5","opponent defensive rating"]}

print("Models ready ✔")

[fallback] frequency-based match model
[fallback] avg-stat top-player model
Models ready ✔


In [7]:
# Quick test — dono models chal rahe hain?
print("Match winner test:")
print(predict_match_winner("Collingwood Magpies", "Geelong Cats"))
print()
print("Top player test:")
print(predict_top_player("Geelong Cats", stat="disposals"))

Match winner test:
{'home_team': 'Collingwood Magpies', 'away_team': 'Geelong Cats', 'home_prob': 0.609, 'away_prob': 0.391, 'top_features': ['historical win rate', 'home advantage']}

Top player test:
{'team': 'Geelong Cats', 'stat': 'disposals', 'predictions': [('Bailey Smith', 31.32), ('Joel Selwood', 24.68), ('Patrick Dangerfield', 24.32)], 'top_features': ['avg disposals last 5', 'opponent defensive rating']}


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME, temperature=0.2,
    google_api_key=os.environ["GEMINI_API_KEY"],
)

class GraphState(TypedDict, total=False):
    query: str
    history: List[Dict[str,str]]
    intent: str
    entities: Dict[str, Any]
    tool_result: Optional[Dict]
    validation: str
    clarification: Optional[str]
    response: str
    trace: List[str]

In [9]:
# Cell 8 (FIXED): Router node — handles list content from Gemini

ROUTER_PROMPT = """You are an intent classifier for an AFL assistant.
Classify into exactly ONE: "prediction","retrieval","factual","off_topic".
- prediction: who will win / top-score / future outcome
- retrieval: past stats, scores, results
- factual: general AFL rules/history
- off_topic: not AFL
Extract: teams[], players[], stat, timeframe.
Return ONLY valid JSON (no markdown, no code fence):
{"intent":"...","teams":[],"players":[],"stat":null,"timeframe":null}
"""

def _content_to_text(content):
    """Normalize Gemini content → plain string"""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for blk in content:
            if isinstance(blk, dict):
                parts.append(blk.get("text", ""))
            else:
                parts.append(str(blk))
        return "".join(parts)
    return str(content)

def _extract_json(text):
    """Extract JSON object from possibly noisy text"""
    text = re.sub(r"```json|```", "", text).strip()
    # find first { ... last }
    start = text.find("{")
    end   = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end+1]
    return text

def router_node(state):
    q = state["query"]
    state.setdefault("trace", []).append(f"[router] query={q!r}")
    try:
        r = llm.invoke([{"role":"system","content":ROUTER_PROMPT},
                        {"role":"user","content":q}])
        raw = _content_to_text(r.content)
        raw = _extract_json(raw)
        data = json.loads(raw)
    except Exception as e:
        state["trace"].append(f"[router] parse err {e}")
        data = {"intent":"off_topic","teams":[],"players":[],"stat":None,"timeframe":None}

    intent = data.get("intent","off_topic")
    low = q.lower()
    if any(k in low for k in ["who will win","who'll win","will win","top-score","top score","who will top"]):
        intent = "prediction"
    elif any(k in low for k in ["stats","last round","how many","score","result"]):
        if intent != "prediction":
            intent = "retrieval"

    state["intent"] = intent
    state["entities"] = {
        "teams": data.get("teams", []) or [],
        "players": data.get("players", []) or [],
        "stat": data.get("stat"),
        "timeframe": data.get("timeframe"),
    }
    state["trace"].append(f"[router] intent={intent} entities={state['entities']}")
    return state

In [10]:
def retrieval_node(state):
    ents = state.get("entities",{})
    teams = [resolve_team(t) for t in ents.get("teams",[])]
    teams = [t for t in teams if t]
    state["trace"].append(f"[retrieval] resolved={teams}")

    if not teams:
        state["tool_result"] = None
        state["validation"] = "needs_clarification"
        state["clarification"] = "Which team's stats? Please name the team."
        return state

    team = teams[0]

    mask = (team_matches["team_name"]==team)
    sub  = team_matches[mask].sort_values("match_date").tail(5)
    if sub.empty:
        state["tool_result"]=None; state["validation"]="error"; return state

    rows=[]
    for _,r in sub.iterrows():
        rows.append({
            "date": str(r["match_date"].date()) if pd.notna(r["match_date"]) else None,
            "team": r["team_name"],
            "opponent": r["opponent"],
            "team_score": int(r["team_score"]) if pd.notna(r["team_score"]) else None,
            "opponent_score": int(r["opponent_score"]) if pd.notna(r["opponent_score"]) else None,
            "result": r["result"],
        })

    top=[]
    if "player_name" in round_stats.columns and "goals" in round_stats.columns:
        ps = (round_stats[round_stats["team"]==team]
              .groupby("player_name")["goals"].sum()
              .dropna().sort_values(ascending=False).head(3))
        top = [(p, int(v)) for p,v in ps.items()]

    state["tool_result"] = {"type":"retrieval","team":team,
                            "recent":rows,"top_scorers":top}
    state["validation"] = "ok"
    state["trace"].append(f"[retrieval] {len(rows)} games, {len(top)} scorers")
    return state

In [11]:
def prediction_node(state):
    ents  = state.get("entities",{})
    q_low = state["query"].lower()

    resolved = [resolve_team(t) for t in ents.get("teams",[])]
    resolved = [t for t in resolved if t]
    state["trace"].append(f"[prediction] resolved={resolved}")

    if len(resolved) < 2:
        for a,c in TEAM_ALIASES.items():
            if a in q_low and c not in resolved:
                resolved.append(c)

    wants_player = any(k in q_low for k in
                       ["top-score","top score","top scorer","most goals","best player"])
    wants_match  = any(k in q_low for k in ["win","beat","winner"])

    # ---- Top player prediction ----
    if wants_player or (not wants_match and len(resolved)==1):
        if not resolved:
            state["validation"]="needs_clarification"
            state["clarification"]="Which team's top scorer would you like me to predict?"
            state["tool_result"]=None; return state
        team = resolved[0]
        stat = (ents.get("stat") or "goals").lower()
        if stat not in ("goals","disposals","marks","kicks","tackles"):
            state["tool_result"]=None; state["validation"]="out_of_scope"
            state["clarification"]=f"I only model goals/disposals/marks/kicks/tackles, not '{stat}'."
            return state
        res = predict_top_player(team, stat=stat)
        if not res:
            state["validation"]="error"; state["tool_result"]=None; return state
        state["tool_result"]={"type":"prediction_top_player",**res}
        state["validation"]="ok"
        state["trace"].append(f"[prediction] top {stat} for {team}")
        return state

    # ---- Match winner ----
    if len(resolved) < 2:
        state["validation"]="needs_clarification"
        state["clarification"]="I need TWO teams to predict a match. e.g. 'Pies vs Cats'."
        state["tool_result"]=None; return state

    home, away = resolved[0], resolved[1]
    res = predict_match_winner(home_team=home, away_team=away)
    if not res:
        state["validation"]="error"; state["tool_result"]=None; return state
    state["tool_result"]={"type":"prediction_match",**res}
    state["validation"]="ok"
    state["trace"].append(f"[prediction] {home} vs {away}")
    return state

In [15]:
# Cell 11 (REQUIRED): Factual, Refusal, Clarification nodes

FACTUAL_SYS = ("You are an AFL facts assistant. Answer concisely. "
               "Never invent statistics.")

def _content_to_text(content):
    """Normalize Gemini content → plain string"""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for blk in content:
            if isinstance(blk, dict):
                parts.append(blk.get("text", ""))
            else:
                parts.append(str(blk))
        return "".join(parts)
    return str(content)

def factual_node(state):
    state["trace"].append("[factual] LLM call")
    r = llm.invoke([{"role":"system","content":FACTUAL_SYS},
                    {"role":"user","content":state["query"]}])
    state["response"] = _content_to_text(r.content)
    state["validation"] = "ok"
    return state

def refusal_node(state):
    state["trace"].append("[refusal]")
    state["response"] = ("I'm an AFL assistant — I can help with AFL facts, stats, "
                         "and predictions. That's outside my scope.")
    state["validation"] = "ok"
    return state

def clarification_node(state):
    state["trace"].append("[clarify]")
    state["response"] = state.get("clarification") or "Could you clarify?"
    return state

print("✅ Cell 11 nodes defined: factual_node, refusal_node, clarification_node, _content_to_text")

✅ Cell 11 nodes defined: factual_node, refusal_node, clarification_node, _content_to_text


In [16]:
def validation_node(state):
    intent = state.get("intent")
    result = state.get("tool_result")
    state.setdefault("validation","ok")
    if intent in ("retrieval","prediction"):
        if result is None and state["validation"]=="ok":
            state["validation"]="error"
        state["trace"].append(f"[validation] {state['validation']}")
    return state

FORMAT_SYS = """You are the response formatter for an AFL assistant.
Rules:
1. PREDICTION → express as probability ("X% chance"), NEVER certainty.
   Add disclaimer "Predictions are probabilistic, not guaranteed."
   List 2-3 grounding features.
2. RETRIEVAL → summarise stats briefly.
3. Keep under 120 words. Do NOT invent data.
"""

def format_node(state):
    state["trace"].append("[format]")
    payload = {"intent":state.get("intent"),
               "tool_result":state.get("tool_result"),
               "validation":state.get("validation")}
    r = llm.invoke([{"role":"system","content":FORMAT_SYS},
                    {"role":"user","content":
                     f"Query: {state['query']}\nTool: {json.dumps(payload)}"}])
    state["response"] = _content_to_text(r.content)   # ← ye line badli
    if state.get("intent")=="prediction" and "probabilis" not in state["response"].lower():
        state["response"] += "\n\n(Predictions are probabilistic, not guaranteed.)"
    state["history"] = state.get("history",[]) + [
        {"role":"user","content":state["query"]},
        {"role":"assistant","content":state["response"]}]
    return state

In [17]:
from langgraph.graph import StateGraph, END

def route_after_router(state):
    return {"prediction":"prediction","retrieval":"retrieval",
            "factual":"factual","off_topic":"refusal"}.get(
                state.get("intent","off_topic"), "refusal")

def route_after_validation(state):
    v = state.get("validation")
    if v=="needs_clarification": return "clarify"
    if v=="out_of_scope":        return "clarify"
    if v=="error":               return "refusal"
    return "format"

b = StateGraph(GraphState)
b.add_node("router",     router_node)
b.add_node("retrieval",  retrieval_node)
b.add_node("prediction", prediction_node)
b.add_node("factual",    factual_node)
b.add_node("refusal",    refusal_node)
b.add_node("clarify",    clarification_node)
b.add_node("validation", validation_node)
b.add_node("format",     format_node)

b.set_entry_point("router")
b.add_conditional_edges("router", route_after_router, {
    "prediction":"prediction","retrieval":"retrieval",
    "factual":"factual","refusal":"refusal"})
b.add_edge("prediction","validation")
b.add_edge("retrieval","validation")
b.add_edge("factual","format")
b.add_edge("refusal","format")
b.add_edge("clarify",END)
b.add_conditional_edges("validation", route_after_validation, {
    "clarify":"clarify","refusal":"refusal","format":"format"})
b.add_edge("format",END)

graph = b.compile()
print("Graph compiled ✔")

Graph compiled ✔


In [18]:
def run_turn(query, history=None):
    state = {"query":query,"history":history or [],"trace":[]}
    return graph.invoke(state)

out = run_turn("Will the Pies beat the Cats this week?")
print("INTENT:", out["intent"])
print("TRACE:")
for t in out["trace"]: print("  ", t)
print("\nRESPONSE:\n", out["response"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


INTENT: prediction
TRACE:
   [router] query='Will the Pies beat the Cats this week?'
   [router] intent=prediction entities={'teams': ['Pies', 'Cats'], 'players': [], 'stat': None, 'timeframe': 'this week'}
   [prediction] resolved=['Collingwood Magpies', 'Geelong Cats']
   [prediction] Collingwood Magpies vs Geelong Cats
   [validation] ok
   [format]

RESPONSE:
 Based on the model, the Collingwood Magpies have a **60.9% chance** of defeating the Geelong Cats (who have a 39.1% chance). 

Key factors supporting this prediction include:
- Home advantage
- Historical win rate

*Predictions are probabilistic, not guaranteed.*


In [19]:
# Cell 15 (FAST): Routing accuracy — only router node called

test_cases = [
    ("Who will win Pies vs Cats this week?","prediction"),
    ("Who will top-score for Richmond this round?","prediction"),
    ("Will Brisbane beat Sydney on the weekend?","prediction"),
    ("Who's going to win the Eagles vs Dockers derby?","prediction"),
    ("Predict the top goal scorer for Carlton.","prediction"),
    ("What were Collingwood's stats last round?","retrieval"),
    ("How many goals did Geelong kick last week?","retrieval"),
    ("Show me Richmond's last 5 results.","retrieval"),
    ("What was the score in the Swans game last round?","retrieval"),
    ("Give me Melbourne's recent stats.","retrieval"),
    ("How many players are on an AFL team?","factual"),
    ("When was the AFL founded?","factual"),
    ("What is a behind worth?","factual"),
    ("How long is a quarter in the AFL?","factual"),
    ("Who is the current AFL CEO?","factual"),
    ("What's the weather in Sydney today?","off_topic"),
    ("Can you book me a flight to Melbourne?","off_topic"),
    ("Write me a poem about cricket.","off_topic"),
    ("Tell me a joke.","off_topic"),
    ("What's the best pizza in Carlton?","off_topic"),
]

rows=[]
for q, exp in test_cases:
    state = {"query": q, "history": [], "trace": []}
    state = router_node(state)          # sirf router — fast
    got = state.get("intent")
    rows.append({"query":q,"expected":exp,"predicted":got,"correct":got==exp})
    print(f"{'✔' if got==exp else '✘'}  exp={exp:10s} got={got:10s} | {q}")

acc_df = pd.DataFrame(rows)
print()
print(acc_df.to_string(index=False))
print(f"\nRouting accuracy: {acc_df['correct'].mean()*100:.1f}%")

✔  exp=prediction got=prediction | Who will win Pies vs Cats this week?
✔  exp=prediction got=prediction | Who will top-score for Richmond this round?
✔  exp=prediction got=prediction | Will Brisbane beat Sydney on the weekend?
✔  exp=prediction got=prediction | Who's going to win the Eagles vs Dockers derby?
✔  exp=prediction got=prediction | Predict the top goal scorer for Carlton.
✔  exp=retrieval  got=retrieval  | What were Collingwood's stats last round?
✔  exp=retrieval  got=retrieval  | How many goals did Geelong kick last week?
✔  exp=retrieval  got=retrieval  | Show me Richmond's last 5 results.
✔  exp=retrieval  got=retrieval  | What was the score in the Swans game last round?
✔  exp=retrieval  got=retrieval  | Give me Melbourne's recent stats.
✘  exp=factual    got=retrieval  | How many players are on an AFL team?
✘  exp=factual    got=off_topic  | When was the AFL founded?
✘  exp=factual    got=off_topic  | What is a behind worth?
✘  exp=factual    got=off_topic  | How long

In [24]:
# Cell 16 — safer version: har node alag call karega naye llm se

import os
from langchain_google_genai import ChatGoogleGenerativeAI

# ⚠️ NAYI key
NEW_API_KEY = ""

llm_backup = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.2,
    google_api_key=NEW_API_KEY,
)

# Custom run_turn jo naye llm se nodes ko chalata hai
def run_turn_with_llm(query, llm_instance, history=None):
    """Same as run_turn but uses a custom llm"""
    # Temporarily swap
    global llm
    _saved = llm
    llm = llm_instance
    try:
        state = {"query": query, "history": history or [], "trace": []}
        return graph.invoke(state)
    finally:
        llm = _saved


conversations = [
    ["Will the Pies beat the Cats this week?"],
    ["Who will top-score for Richmond this round?"],
    ["What were Collingwood's stats last round?"],
    ["How many players are on an AFL team?"],
    ["What's the weather in Sydney?"],
    ["Will Collingwood win?"],
    ["Who will win this week?"],
    ["Predict the top tackler for Geelong this week."],
    ["Will the Pies beat the Cats this week?",
     "What about last time they played?"],
    ["How many goals did Hawkins kick last round?"],
]

for i, turns in enumerate(conversations, 1):
    print(f"\n===== Conversation {i} =====")
    history = []
    for q in turns:
        try:
            out = run_turn_with_llm(q, llm_backup, history)
            history = out.get("history", history)
            print(f"USER: {q}")
            print(f"INTENT: {out.get('intent')} | VALIDATION: {out.get('validation')}")
            print(f"RESPONSE: {out.get('response')[:250]}...\n")
        except Exception as e:
            print(f"❌ Error on '{q}': {e}\n")
            break

print("\n Cell 16 done.")


===== Conversation 1 =====


C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: Will the Pies beat the Cats this week?
INTENT: prediction | VALIDATION: ok
RESPONSE: Collingwood (Pies) have a **60.9% chance** of defeating the Geelong Cats this week, while Geelong has a **39.1% chance** of winning.

Key factors influencing this model prediction include:
* Home advantage
* Historical win rate

*Predictions are prob...


===== Conversation 2 =====


C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: Who will top-score for Richmond this round?
INTENT: prediction | VALIDATION: out_of_scope
RESPONSE: I only model goals/disposals/marks/kicks/tackles, not 'top-score'....


===== Conversation 3 =====


...

USER: Will the Pies beat the Cats this week?
INTENT: prediction | VALIDATION: ok
RESPONSE: Collingwood has a 61% chance of beating Geelong this week, while Geelong holds a 39% chance of victory. 

Key factors influencing this prediction include:
- Historical win rate
- Home advantage

*Predictions are probabilistic, not guaranteed.*...



C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: What about last time they played?
INTENT: retrieval | VALIDATION: needs_clarification
RESPONSE: Which team's stats? Please name the team....


===== Conversation 10 =====


C:\Users\cfiza\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


USER: How many goals did Hawkins kick last round?
INTENT: retrieval | VALIDATION: needs_clarification
RESPONSE: Which team's stats? Please name the team....


 Cell 16 done.


In [23]:
comparison = """
Comparing this LangGraph orchestration to a single monolithic LangChain agent:

A single monolithic agent must decide intent, call tools, and format responses
inside one prompt, which makes it easy for prediction answers to slip through
without the required probabilistic disclaimer. By splitting the flow into
router → tool → validation → format nodes, this design guarantees every
prediction passes through a formatting node that enforces probability framing
and grounding features. Validation and fallback branches also become first-class:
ambiguous inputs route to clarification instead of hallucinating a fixture, and
out-of-scope stats are refused cleanly.
"""
print(comparison)


Comparing this LangGraph orchestration to a single monolithic LangChain agent:

A single monolithic agent must decide intent, call tools, and format responses
inside one prompt, which makes it easy for prediction answers to slip through
without the required probabilistic disclaimer. By splitting the flow into
router → tool → validation → format nodes, this design guarantees every
prediction passes through a formatting node that enforces probability framing
and grounding features. Validation and fallback branches also become first-class:
ambiguous inputs route to clarification instead of hallucinating a fixture, and
out-of-scope stats are refused cleanly.

